In [ ]:
import logging
import gc


import torch
import cv2
import numpy as np
from tqdm import tqdm
from scipy.interpolate import CubicSpline

from decord import VideoReader
from decord._ffi.base import DECORDError

import subprocess
import cupy as cp
from cupyx.scipy.ndimage import zoom, gaussian_filter
from ultralytics import YOLO

device = "cuda" if torch.cuda.is_available() else "cpu"
model_path = "backend/models/yolov12n-face.pt"
model_tracking = YOLO(model_path).to(device).eval()

logger = logging.getLogger(__name__)

def tracking_video(track_per_second: int, smoothing_factor: float, batch_size: int):
    # vr = VideoReader(f'{FILES_DIR}/sources/{project.video_filename}', ctx=decord.gpu(0))
    vr = VideoReader(f'{FILES_DIR}/sources/{project.video_filename}')
    fps = vr.get_avg_fps()
    total_frames = len(vr)
    width, height = vr[0].shape[1], vr[0].shape[0]

    track_every_n = max(int(fps // track_per_second), 1)

    logger.info(f"Tracking: Video loaded (fps={fps}, width={width}, track_every_n={track_every_n}, frames={total_frames})")


    indices = list(range(0, total_frames, track_every_n))


    centers = {}
    for i in tqdm(range(0, len(indices), batch_size)):
        batch_indices = indices[i:i+batch_size]

        try:
            batch = vr.get_batch(batch_indices).asnumpy()
        except DECORDError as e:
            logger.warning(f"Decord failed at batch {i} ({batch_indices}): {e}")
            frames = []
            for idx in batch_indices:
                try:
                    frames.append(vr[idx].asnumpy())
                except DECORDError as e2:
                    logger.error(f"Frame {idx} failed: {e2}")
                    frames.append(np.zeros((height, width, 3), dtype=np.uint8))
            batch = np.stack(frames, axis=0)

        batch = [cv2.cvtColor(f, cv2.COLOR_RGB2BGR) for f in batch]

        with torch.no_grad():
            results = models_loader.model_tracking(batch, verbose=False)

        for j, result in enumerate(results):
            idx = indices[i + j]
            if result.boxes and result.boxes.xyxy.numel() > 0:
                x1, y1, x2, y2 = result.boxes.xyxy[0].tolist()
                cx = int((x1 + x2) / 2)
            else:
                cx = width // 2
            centers[idx] = cx
        
    del batch, results
    torch.cuda.empty_cache()
    gc.collect()
            
    
    centers_x = np.zeros(total_frames, dtype=np.float32)
    for idx, cx in centers.items():
        centers_x[idx] = cx

    known_indices = np.array(sorted(centers.keys()))
    known_values = np.array([centers[i] for i in known_indices])

    all_indices = np.arange(total_frames)
#     # centers_x = np.interp(all_indices, known_indices, known_values, left=known_values[0], right=known_values[-1])
    cs = CubicSpline(known_indices, known_values, bc_type='clamped')
    centers_x = cs(all_indices)
    centers_x[:known_indices[0]] = known_values[0]
    centers_x[known_indices[-1]+1:] = known_values[-1]


    smoothed = np.empty_like(centers_x)
    smoothed[0] = centers_x[0]
    alpha = smoothing_factor
    for i in range(1, len(centers_x)):
        smoothed[i] = smoothed[i-1] + alpha * (centers_x[i] - smoothed[i-1])

    del vr, centers_x, known_indices, known_values, all_indices, cs
    gc.collect()

    return smoothed.astype(int).tolist()